#  Модуль 9. Масштабирование и нормализация

## Подробный конспект

### 9.1. Зачем приводить признаки к одному масштабу

Представьте, что у вас два признака: **возраст** (18–90 лет) и **доход** (20 000–500 000 ₽. Если посчитать расстояние между двумя клиентами, доход «задавит» возраст: разница в 100 000 ₽ весомее разницы в 10 лет.

Модели, основанные на **расстояниях** или **градиентном спуске**, воспринимают признаки «неравноправно»:

- **KNN, KMeans:** ближайшие соседи определяются в основном по доходу, возраст почти не влияет.
- **SVM:** разделяющая граница сильно сдвинута в сторону дохода.
- **Линейная/логистическая регрессия, нейросети:** градиентный спуск сходится медленно и «зигзагообразно», если признаки разного масштаба.
- **Регуляризация (L1, L2):** штрафует большие веса. Если один признак в тысячи раз больше другого, его коэффициент естественно будет меньше — регуляризация исказит картину.

**Цель:** сделать так, чтобы все числовые признаки вносили примерно равный вклад в модель, независимо от их исходных единиц измерения.

### 9.2. Когда масштабирование нужно, а когда нет

| Тип модели | Нужно ли масштабирование? | Почему |
|-----------|--------------------------|--------|
| **KNN, KMeans** | TRUE Обязательно | Расстояния искажаются доминирующими признаками |
| **SVM** | TRUE Обязательно | Ядро зависит от расстояний |
| **Линейная/логистическая регрессия** | TRUE Обязательно | Градиентный спуск + регуляризация |
| **Нейросети** | TRUE Обязательно | Градиентный спуск, инициализация весов |
| **PCA, t-SNE, кластеризация** | TRUE Обязательно | Зависят от дисперсии и ковариации |
| **Деревья решений** | FALSE Не нужно | Делят по порогам, инвариантны к монотонным преобразованиям |
| **Случайный лес, XGBoost, LightGBM, CatBoost** | FALSE Не нужно | Базовые алгоритмы — деревья |
| **Наивный Байес** | FALSE Не нужно | Вероятности считаются независимо по признакам |

> **Практическое правило:** если модель использует расстояния или градиентный спуск — масштабируйте. Если деревья — можно не тратить время.

### 9.3. StandardScaler (Z-нормализация)

Самый популярный метод. Приводит данные к виду: **среднее ≈ 0, стандартное отклонение ≈ 1**.

**Формула:**
$$z = \frac{x - \mu}{\sigma}$$

Где $\mu$ — среднее, $\sigma$ — стандартное отклонение.

In [1]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Исходные данные
age = np.array([[25], [30], [35], [40], [45], [50], [100]])  # 100 — выброс

scaler = StandardScaler()
age_scaled = scaler.fit_transform(age)

print(f"Среднее до: {age.mean():.2f}, std: {age.std():.2f}")
print(f"Среднее после: {age_scaled.mean():.4f}, std: {age_scaled.std():.4f}")
# Среднее ≈ 0, std ≈ 1

Среднее до: 46.43, std: 23.26
Среднее после: -0.0000, std: 1.0000


**Интерпретация:** значение +1.5 означает «на 1.5 стандартных отклонения выше среднего».

**Плюсы:**
- Интерпретируемость (Z-score)
- Не меняет форму распределения (только сдвигает и растягивает)
- Работает хорошо при нормальном или близком к нормальному распределении

**Минусы:**
- **Чувствителен к выбросам.** Один выброс в 1 000 000 «растянет» стандартное отклонение, и весь масштаб исказится.
- Не ограничивает диапазон (значения могут быть −3, +5 и т.д.)

**Когда использовать:** данные без сильных выбросов, распределение примерно симметрично.

### 9.4. MinMaxScaler (Мин-макс нормализация)

Приводит данные к заданному диапазону, обычно **[0, 1]** или **[−1, 1]**.

**Формула:**
$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [2]:
from sklearn.preprocessing import MinMaxScaler

доход = np.array([[30000], [50000], [60000], [80000], [150000]])

scaler = MinMaxScaler(feature_range=(0, 1))
доход_scaled = scaler.fit_transform(доход)

print(доход_scaled.flatten())
# [0.   0.167 0.25  0.417 1.   ]

[0.         0.16666667 0.25       0.41666667 1.        ]


**Плюсы:**
- Чётко заданные границы (удобно для нейросетей, где активации ожидают [0,1] или [−1,1])
- Сохраняет нулевые значения (если 0 в данных, оно останется 0)

**Минусы:**
- **Очень чувствителен к выбросам.** Если добавить доход 10 000 000, весь диапазон [30K–150K] сожмётся в крошечный отрезок около 0.
- Не центрирован (среднее не 0, если распределение скошено)

**Когда использовать:**
- Нейросети (sigmoid, tanh на входе/выходе)
- Когда нужен строго определённый диапазон
- Данные уже очищены от выбросов

### 9.5. RobustScaler (Устойчивое масштабирование)

Использует **медиану** и **IQR** вместо среднего и стандартного отклонения. Устойчив к выбросам.

**Формула:**
$$x_{scaled} = \frac{x - \text{медиана}}{\text{IQR}}$$

Где $\text{IQR} = Q3 - Q1$.

In [3]:
from sklearn.preprocessing import RobustScaler

# Данные с выбросом
доход = np.array([[30000], [45000], [50000], [55000], [60000], [5000000]])  # выброс

scaler = RobustScaler()
доход_scaled = scaler.fit_transform(доход)

print(доход_scaled.flatten())
# Основная масса данных останется в разумных пределах вокруг 0,
# а выброс будет просто очень большим числом, но не испортит масштаб

[-1.800e+00 -6.000e-01 -2.000e-01  2.000e-01  6.000e-01  3.958e+02]


**Плюсы:**
- Не чувствителен к выбросам
- Хорошо работает с реальными, «грязными» данными

**Минусы:**
- Не гарантирует диапазон (значения могут выходить за [−3, 3])
- Меньше «стандартная» интерпретация, чем у StandardScaler

**Когда использовать:**
- Данные содержат выбросы, которые вы не хотите/не можете удалять
- EDA-подготовка перед моделями, чувствительными к расстояниям

### 9.6. MaxAbsScaler

Делит каждое значение на **максимальное по модулю** значение в столбце. Результат лежит в **[−1, 1]**.

**Формула:**
$$x_{scaled} = \frac{x}{|x_{max}|}$$

In [4]:
from sklearn.preprocessing import MaxAbsScaler

# Разреженные данные (много нулей)
данные = np.array([[0, 0, 100],
                   [0, 50, 0],
                   [10, 0, 0]])

scaler = MaxAbsScaler()
данные_scaled = scaler.fit_transform(данные)

print(данные_scaled)
# [[0.  0.  1. ]
#  [0.  1.  0. ]
#  [1.  0.  0. ]]

[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]


**Ключевая особенность:** **сохраняет нули.** Если данные разреженные (sparse matrix, например, TF-IDF или OHE с множеством нулей), MaxAbsScaler не создаёт новых ненулевых элементов.

**Когда использовать:**
- Разреженные матрицы (sparse data)
- Когда нули несут смысл «отсутствия признака» и их не должно стать −0.7 после StandardScaler

### 9.7. Normalizer (нормализация векторов)

**Критически важно:** Normalizer работает **по строкам (наблюдениям)**, а не по столбцам (признакам)!

Все предыдущие скейлеры обрабатывали каждый признак отдельно: считали средний доход по всем строкам и масштабировали столбец «доход». Normalizer берёт **одну строку** и делит все её признаки на норму вектора.

**Формула (L2-норма):**
$$x_{normalized} = \frac{x}{\sqrt{x_1^2 + x_2^2 + ... + x_n^2}}$$

После этого длина вектора каждой строки станет равна 1.

In [5]:
from sklearn.preprocessing import Normalizer

# Каждая строка — вектор признаков одного объекта
X = np.array([[1, 2, 3],
              [4, 5, 6],
              [0, 0, 10]])

normalizer = Normalizer(norm='l2')  # евклидова норма
X_norm = normalizer.fit_transform(X)

# Проверим длину первого вектора
np.linalg.norm(X_norm[0])  # 1.0

np.float64(1.0)

**Варианты норм:**
- `'l2'` — евклидова (по умолчанию). Полезна, когда важны углы между векторами.
- `'l1'` — сумма модулей = 1. Полезна, когда важны пропорции.
- `'max'` — деление на максимум по модулю в строке.

**Когда использовать:**
- **Текстовые данные** (TF-IDF, Bag of Words): длина документа не должна влиять, важны пропорции слов.
- **Косинусное сходство:** когда нужно сравнивать направление векторов, а не их длину.
- **Задачи с геометрической интерпретацией** векторов.

**Когда НЕ использовать:**
- Обычные табличные данные, где важны абсолютные значения признаков.
- Если признаки имеют разную природу (возраст, доход, количество детей) — нормирование строки лишает смысла масштаб признаков.

### 9.8. Сравнение всех методов

| Метод | Формула | Диапазон | Устойчив к выбросам | Работает с sparse | Центрирует | Когда использовать |
|-------|---------|----------|---------------------|-------------------|------------|-------------------|
| **StandardScaler** | $(x - \mu) / \sigma$ | Не ограничен | FALSE Нет | FALSE Нет | TRUE Да | Нет выбросов, нормальное распределение |
| **MinMaxScaler** | $(x - min) / (max - min)$ | [0, 1] | FALSE Нет | FALSE Нет | FALSE Нет | Нейросети, нужен строгий диапазон, нет выбросов |
| **RobustScaler** | $(x - median) / IQR$ | Не ограничен | TRUE Да | FALSE Нет | TRUE Да | Есть выбросы, «грязные» данные |
| **MaxAbsScaler** | $x / |max|$ | [−1, 1] | FALSE Нет (но сохраняет нули) | TRUE Да | FALSE Нет | Разреженные данные (sparse) |
| **Normalizer** | $x / ||x||$ | [−1, 1] (по строке) | TRUE Да (по строке) | TRUE Да | — | Тексты, косинусное сходство, векторная модель |

### 9.9. Data Leakage при масштабировании

**Самая опасная ошибка новичков.**

**Неправильно:**

In [ ]:
# FALSE НЕПРАВИЛЬНО: утечка данных
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # fit на ВСЕХ данных

X_train, X_test = train_test_split(X_scaled, test_size=0.2)

**Почему это утечка:** scaler «увидел» тестовые данные при подсчёте среднего и стандартного отклонения. Тестовые данные «просочились» в обучение.

**Правильно:**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

# TRUE Правильно: fit только на train, transform на train и test
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # только transform!

**Правило:** любая трансформация, обучаемая на данных (scaler, encoder, imputer, PCA), должна обучаться (`fit`) **только на обучающей выборке**. К тестовой и продакшен-данным применяется уже обученная трансформация (`transform`).

### 9.10. Сохранение и обратное преобразование

Если вы масштабировали таргет (например, логарифмировали или стандартизировали цену), предсказания модели нужно будет перевести обратно.

In [ ]:
scaler = StandardScaler()
y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1))

# ... обучение модели ...
y_pred_scaled = model.predict(X_test_scaled)

# Обратно в исходные единицы
y_pred = scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))

**Для признаков:** обратное преобразование нужно, если вы хотите интерпретировать коэффициенты модели в исходных единицах.

In [ ]:
# Пример: вес линейной регрессии в масштабированных признаках
coef_scaled = model.coef_

# В исходных единицах: нужно учитывать масштаб
coef_original = coef_scaled / scaler.scale_

### 9.11. Практический пример: полный цикл масштабирования

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

np.random.seed(42)

# Создаём данные с разными масштабами и выбросами
df = pd.DataFrame({
    'возраст': np.random.normal(40, 12, 1000).clip(18, 90),
    'доход': np.random.lognormal(11, 0.6, 1000),  # правый хвост + выбросы
    'стаж': np.random.poisson(10, 1000),
    'оценка_кредита': np.random.normal(700, 50, 1000).clip(300, 850)
})

# Добавим выброс в доход
df.loc[np.random.choice(df.index, 20), 'доход'] *= 5

print("Исходная статистика:")
print(df.describe().round(2))

# === РАЗДЕЛЕНИЕ ===
X = df
y = np.random.normal(0, 1, 1000)  # фиктивный таргет

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# === MASSШТАБИРОВАНИЕ ===
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

results = {}

for name, scaler in scalers.items():
    # Fit только на train!
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    results[name] = {
        'train': X_train_s,
        'test': X_test_s,
        'scaler': scaler
    }

# === ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (name, data) in enumerate(results.items()):
    ax1 = axes[0, idx]
    ax2 = axes[1, idx]
    
    # Гистограмма дохода
    ax1.hist(data['train'][:, 1], bins=50, alpha=0.7, label='train')
    ax1.hist(data['test'][:, 1], bins=50, alpha=0.7, label='test')
    ax1.set_title(f'{name}\n(доход)')
    ax1.legend()
    
    # Boxplot всех признаков
    ax2.boxplot(data['train'], labels=['возраст', 'доход', 'стаж', 'кредит'])
    ax2.set_title(f'{name}\n(все признаки)')

plt.tight_layout()
plt.show()

# === ПРОВЕРКА НА ВЫБРОСЫ ===
print("\nДиапазон дохода после масштабирования (трейн):")
for name, data in results.items():
    доход_col = data['train'][:, 1]  # столбец дохода
    print(f"{name}: min={доход_col.min():.2f}, max={доход_col.max():.2f}")

# RobustScaler сохранит выброс большим, но не «взорвёт» масштаб
# MinMaxScaler сожмёт всё остальное в крошечный диапазон из-за выброса

### 9.12. Чек-лист для самопроверки

Перед переходом к Модулю 10 убедитесь, что вы:

- [ ] Понимаете, зачем масштабировать признаки (равноправие в расстояниях и градиентном спуске)
- [ ] Знаете, какие модели требуют масштабирования, а какие — нет
- [ ] Можете применить StandardScaler и объяснить его формулу
- [ ] Понимаете, почему StandardScaler чувствителен к выбросам
- [ ] Умеете использовать MinMaxScaler и знаете его диапазон
- [ ] Знаете, когда MinMaxScaler опасен (наличие выбросов)
- [ ] Понимаете принцип RobustScaler и почему он устойчив к аномалиям
- [ ] Знаете, зачем нужен MaxAbsScaler (разреженные данные)
- [ ] Чётко различаете Normalizer (по строкам) и остальные скейлеры (по столбцам)
- [ ] Понимаете, что такое Data Leakage при масштабировании
- [ ] Умеете правильно: `fit_transform` на train, `transform` на test
- [ ] Знаете, как сделать `inverse_transform` для возврата к исходным единицам

> **Переход к Модулю 10:** Теперь, когда вы умеете масштабировать и кодировать признаки, пора научиться их **создавать**. Следующий модуль посвящён Feature Engineering — извлечению максимума информации из имеющихся данных: комбинации признаков, временные признаки, текст, геоданные и доменные знания.